In [18]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# PROJECT PATHS

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
ROLE_RADAR_DIR = RAW_DIR / "role_radar"
TRAINING_DIR = DATA_DIR / "training"
MODELS_DIR = PROJECT_ROOT / "models"

# Create required directories
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root      :", PROJECT_ROOT)
print("Role Radar        :", ROLE_RADAR_DIR)
print("Training directory:", TRAINING_DIR)
print("Models directory  :", MODELS_DIR)

Project root      : C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence
Role Radar        : C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\raw\role_radar
Training directory: C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\training
Models directory  : C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\models


In [19]:
required_files = [
    "scraped_jobs.json",
    "synthetic_profiles.json",
    "phase3_pairs.json",
    "phase3_labels.json",
    "gold_labels.json"
]

print("RAW DATA FILE CHECK")

all_files_exist = True

for filename in required_files:
    file_path = ROLE_RADAR_DIR / filename

    if file_path.exists():
        print(f"✓ {filename}")
    else:
        print(f"✗ {filename} NOT FOUND")
        all_files_exist = False

print("\nAll required files available:", all_files_exist)

RAW DATA FILE CHECK
✓ scraped_jobs.json
✓ synthetic_profiles.json
✓ phase3_pairs.json
✓ phase3_labels.json
✓ gold_labels.json

All required files available: True


In [20]:
# LOAD RAW JSON DATA

def load_json(filename):
    file_path = ROLE_RADAR_DIR / filename

    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


jobs = load_json("scraped_jobs.json")
profiles = load_json("synthetic_profiles.json")
pairs = load_json("phase3_pairs.json")
labels = load_json("phase3_labels.json")
gold_labels = load_json("gold_labels.json")

# Convert to DataFrames
jobs_df = pd.DataFrame(jobs)
profiles_df = pd.DataFrame(profiles)
pairs_df = pd.DataFrame(pairs)
labels_df = pd.DataFrame(labels)
gold_labels_df = pd.DataFrame(gold_labels)

print("RAW DATA LOADED")

print("Jobs        :", jobs_df.shape)
print("Profiles    :", profiles_df.shape)
print("Pairs       :", pairs_df.shape)
print("Labels      :", labels_df.shape)
print("Gold labels :", gold_labels_df.shape)

RAW DATA LOADED
Jobs        : (2500, 16)
Profiles    : (640, 11)
Pairs       : (5000, 8)
Labels      : (22465, 11)
Gold labels : (108, 13)


In [21]:
# REMOVE DUPLICATE JOBS

jobs_unique_df = jobs_df.drop_duplicates(
    subset="id",
    keep="first"
).copy()

print("Original jobs :", len(jobs_df))
print("Unique jobs   :", len(jobs_unique_df))
print("Removed       :", len(jobs_df) - len(jobs_unique_df))


# PREPARE CANDIDATE PROFILES

profiles_selected = profiles_df[
    [
        "profile_id",
        "roles",
        "skills_primary",
        "skills_secondary",
        "experience_years",
        "seniority",
        "domains",
        "preferences",
        "career_intent",
        "dealbreakers",
        "data_origin"
    ]
].copy()

profiles_selected = profiles_selected.rename(
    columns={
        "profile_id": "candidate_id",
        "roles": "candidate_roles",
        "skills_primary": "candidate_skills_primary",
        "skills_secondary": "candidate_skills_secondary",
        "experience_years": "candidate_experience_years",
        "seniority": "candidate_seniority",
        "domains": "candidate_domains",
        "preferences": "candidate_preferences",
        "career_intent": "candidate_career_intent",
        "dealbreakers": "candidate_dealbreakers",
        "data_origin": "candidate_data_origin"
    }
)


# PREPARE PAIRS

pairs_clean = pairs_df.copy()

pairs_clean = pairs_clean.rename(
    columns={
        "profile_id": "candidate_id"
    }
)


# PREPARE JOBS

jobs_selected = jobs_unique_df[
    [
        "id",
        "title",
        "company",
        "location",
        "description",
        "seniority_level",
        "employment_type",
        "job_function",
        "industry",
        "url",
        "posted_at",
        "scraped_at",
        "role_family_hint",
        "domain_hint",
        "experience_years_hint",
        "remote_hint"
    ]
].copy()

jobs_selected = jobs_selected.rename(
    columns={
        "id": "job_id",
        "title": "job_title",
        "company": "job_company",
        "location": "job_location",
        "description": "job_description",
        "seniority_level": "job_seniority_level",
        "employment_type": "job_employment_type",
        "job_function": "job_function",
        "industry": "job_industry",
        "url": "job_url",
        "posted_at": "job_posted_at",
        "scraped_at": "job_scraped_at",
        "role_family_hint": "job_role_family_hint",
        "domain_hint": "job_domain_hint",
        "experience_years_hint": "job_experience_years_hint",
        "remote_hint": "job_remote_hint"
    }
)

print("\nCandidate profiles:", profiles_selected.shape)
print("Candidate-job pairs:", pairs_clean.shape)
print("Unique jobs:", jobs_selected.shape)

Original jobs : 2500
Unique jobs   : 2471
Removed       : 29

Candidate profiles: (640, 11)
Candidate-job pairs: (5000, 8)
Unique jobs: (2471, 16)


In [22]:
# MERGE CANDIDATES + PAIRS + JOBS

pairs_with_profiles = pairs_clean.merge(
    profiles_selected,
    on="candidate_id",
    how="left",
    validate="many_to_one"
)

training_raw_df = pairs_with_profiles.merge(
    jobs_selected,
    on="job_id",
    how="left",
    validate="many_to_one"
)

print("MERGED DATASET")
print("Shape:", training_raw_df.shape)

print("\nMissing candidate roles:",
      training_raw_df["candidate_roles"].isna().sum())

print("Missing candidate skills:",
      training_raw_df["candidate_skills_primary"].isna().sum())

print("Missing job title:",
      training_raw_df["job_title"].isna().sum())

print("Missing job description:",
      training_raw_df["job_description"].isna().sum())

MERGED DATASET
Shape: (5000, 33)

Missing candidate roles: 0
Missing candidate skills: 0
Missing job title: 0
Missing job description: 0


In [23]:
# SELECT FEATURES FOR ML PIPELINE

feature_source_columns = [
    "pair_id",
    "candidate_id",
    "job_id",
    "pair_type",

    # Candidate
    "candidate_roles",
    "candidate_skills_primary",
    "candidate_skills_secondary",
    "candidate_experience_years",
    "candidate_seniority",
    "candidate_domains",
    "candidate_career_intent",

    # Job
    "job_title",
    "job_description",
    "job_location",
    "job_seniority_level",
    "job_employment_type",
    "job_function",
    "job_industry",
    "job_role_family_hint",
    "job_domain_hint",
    "job_experience_years_hint",
    "job_remote_hint"
]

feature_source_df = training_raw_df[
    feature_source_columns
].copy()


# CREATE BINARY TARGET

feature_source_df["match_label"] = (
    feature_source_df["pair_type"]
    .map({
        "positive": 1,
        "negative": 0,
        "hard_negative": 0
    })
)


# REMOVE COMPLETELY EMPTY COLUMNS

empty_columns = [
    column
    for column in feature_source_df.columns
    if feature_source_df[column].isna().all()
]

if empty_columns:
    feature_source_df = feature_source_df.drop(
        columns=empty_columns
    )


print("CLEAN DATASET")

print("Shape:", feature_source_df.shape)

print("\nRemoved completely empty columns:")
print(empty_columns)

print("\nTarget distribution:")
print(feature_source_df["match_label"].value_counts())

print("\nTarget percentages:")
print(
    feature_source_df["match_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

CLEAN DATASET
Shape: (5000, 21)

Removed completely empty columns:
['job_domain_hint', 'job_experience_years_hint']

Target distribution:
match_label
0    3500
1    1500
Name: count, dtype: int64

Target percentages:
match_label
0    70.0
1    30.0
Name: proportion, dtype: float64


In [24]:
# FINAL PREPROCESSING VALIDATION

print("DATA VALIDATION")

print("Total rows:", len(feature_source_df))

print(
    "Unique pair IDs:",
    feature_source_df["pair_id"].nunique()
)

print(
    "Duplicate pair IDs:",
    feature_source_df["pair_id"].duplicated().sum()
)

print(
    "Unique candidates:",
    feature_source_df["candidate_id"].nunique()
)

print(
    "Unique jobs:",
    feature_source_df["job_id"].nunique()
)

print(
    "Missing target:",
    feature_source_df["match_label"].isna().sum()
)

print("\nMissing values:")
missing = feature_source_df.isna().sum()
print(missing[missing > 0])

print("\nData types:")
print(feature_source_df.dtypes)

DATA VALIDATION
Total rows: 5000
Unique pair IDs: 5000
Duplicate pair IDs: 0
Unique candidates: 581
Unique jobs: 1623
Missing target: 0

Missing values:
job_function    18
dtype: int64

Data types:
pair_id                          str
candidate_id                   int64
job_id                           str
pair_type                        str
candidate_roles               object
candidate_skills_primary      object
candidate_skills_secondary    object
candidate_experience_years     int64
candidate_seniority              str
candidate_domains             object
candidate_career_intent          str
job_title                        str
job_description                  str
job_location                     str
job_seniority_level              str
job_employment_type              str
job_function                     str
job_industry                     str
job_role_family_hint             str
job_remote_hint                 bool
match_label                    int64
dtype: object


In [25]:
# SAVE CLEANED DATASET

CSV_OUTPUT = TRAINING_DIR / "candidate_job_cleaned.csv"
PICKLE_OUTPUT = TRAINING_DIR / "candidate_job_cleaned.pkl"

# Save CSV for easy inspection
feature_source_df.to_csv(
    CSV_OUTPUT,
    index=False
)

# Save Pickle to preserve Python list columns
feature_source_df.to_pickle(
    PICKLE_OUTPUT
)

print("PREPROCESSING COMPLETE")
print("CSV saved:")
print(CSV_OUTPUT)

print("\nPickle saved:")
print(PICKLE_OUTPUT)

print("\nFinal shape:")
print(feature_source_df.shape)

PREPROCESSING COMPLETE
CSV saved:
C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\training\candidate_job_cleaned.csv

Pickle saved:
C:\Users\vramy\OneDrive\Desktop\samba shiva reddy\AI-Resume-Intelligence\data\training\candidate_job_cleaned.pkl

Final shape:
(5000, 21)


In [26]:
# VERIFY OUTPUT

print("OUTPUT VERIFICATION")

print("CSV exists   :", CSV_OUTPUT.exists())
print("Pickle exists:", PICKLE_OUTPUT.exists())

# Reload Pickle
check_df = pd.read_pickle(PICKLE_OUTPUT)

print("\nReloaded shape:", check_df.shape)

print("\nReloaded target distribution:")
print(check_df["match_label"].value_counts())

print("\nExample candidate skills:")
print(check_df["candidate_skills_primary"].iloc[0])

print("\nExample job title:")
print(check_df["job_title"].iloc[0])

OUTPUT VERIFICATION
CSV exists   : True
Pickle exists: True

Reloaded shape: (5000, 21)

Reloaded target distribution:
match_label
0    3500
1    1500
Name: count, dtype: int64

Example candidate skills:
['Talent Acquisition', 'HRIS', 'Employee Relations']

Example job title:
Recruiter
